# Load model and dataset

In [1]:
! pip3 install torch --index-url https://download.pytorch.org/whl/

Looking in indexes: https://download.pytorch.org/whl/
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/1.9 GB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 GB 26.0 MB/s eta 0:01:14
   ---------------------------------------- 0.0/1.9 GB 17.5 MB/s eta 0:01:50
   ---------------------------------------- 0.0/1.9 GB 21.2 MB/s eta 0:01:30
    --------------------------------------- 0.0/1.9 GB 22.9 MB/s eta 0:01:23
    --------------------------------------- 0.0/1.9 GB 25.9 MB/s eta 0:01:13
    --------------------------------------- 0.0/1.9 GB 27.5 MB/s eta 0:01:09
   - ------------


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install pytorch_lightning nervaluate seqeval


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
!pip3 install -r "D:\univ_subj\M_An_I\Sem 2\LLMs for NLP\RoDi\performance_analysis\evaluate\requirements.txt" -q


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import torch
import json
import numpy as np
# from seqeval.metrics import classification_report
from evaluate.evaluate import *

# ── Load best checkpoint ────────────────────────────────────
checkpoints_path = './performance_analysis/checkpoints/'
datasets_path = './performance_analysis/datasets/'
m_model_name = 'bert-base-multilingual-cased'
ro_model_name = 'dumitrescustefan/bert-base-romanian-cased-v1'

ModuleNotFoundError: No module named 'pytorch_lightning'

In [ ]:
with open(datasets_path + 'diac/train.json', "r", encoding="utf8") as f:
    train_diac = json.load(f)

In [ ]:
tags_diac = ["O"] * 16  # tags without the B- or I- prefix
bio2tags_diac = ["O"] * 31  # tags with the B- and I- prefix, all tags are here

for instance in train_diac:
    for tag, tag_index in zip(instance["ner_tags"], instance["ner_ids"]):
        bio2tags_diac[tag_index] = tag  # put the bio2 tag in its correct position
        if tag_index % 2 == 0 and tag_index > 0:
            tags_diac[int(tag_index / 2)] = tag[2:]

In [5]:
best_ckpt = checkpoints_path + 'm_diac/epoch=2.ckpt'
print(f"Best checkpoint: {best_ckpt}")

m_diac_model = TransformerModel.load_from_checkpoint(
    best_ckpt,
    weights_only=True,
    model_name=m_model_name,        # e.g., "bert-base-multilingual-cased"
    tokenizer=AutoTokenizer.from_pretrained(m_model_name),
    lr=2e-05,
    lr_factor=2/3,
    lr_patience=5,
    model_max_length=512,
    bio2tags=your_bio2tags_dict,
    tag_list=your_tag_list
)
m_diac_model.eval()
m_diac_model.cuda()

NameError: name 'checkpoints_path' is not defined

In [ ]:

best_ckpt = checkpoints_path + 'm_nodiac/epoch=3_315.ckpt'
print(f"Best checkpoint: {best_ckpt}")

m_nodiac_model = TransformerModel.load_from_checkpoint(best_ckpt)
m_nodiac_model.eval()
m_nodiac_model.cuda()

In [ ]:
best_ckpt = checkpoints_path + 'ro_diac/epoch=1_315.ckpt'
print(f"Best checkpoint: {best_ckpt}")

ro_diac_model = TransformerModel.load_from_checkpoint(best_ckpt)
ro_diac_model.eval()
ro_diac_model.cuda()

In [ ]:
best_ckpt = checkpoints_path + 'ro_nodiac/epoch=5_315.ckpt'
print(f"Best checkpoint: {best_ckpt}")

ro_nodiac_model = TransformerModel.load_from_checkpoint(best_ckpt)
ro_nodiac_model.eval()
ro_nodiac_model.cuda()

# Crossed Inference

In [ ]:
# Configs
batch_size = 8

# "robert" | "mbert"
MODEL_KEY  = "mbert"
# "diac" | "nodiac"
TRAIN_COND = "nodiac"

RUN_NAME = f"{MODEL_KEY}_train_{TRAIN_COND}"

MODEL_HUB = {
    "robert": "dumitrescustefan/bert-base-romanian-cased-v1",
    "mbert":  "bert-base-multilingual-cased",
}[MODEL_KEY]

# SEED = 315
SEED = 222

BASE     = f"/content/drive/MyDrive/rodi_study/{RUN_NAME}"
CKPT_DIR = f"{BASE}/checkpoints"
RES_DIR  = f"{BASE}/results"
PRED_DIR = f"{BASE}/predictions"


In [ ]:
label_list = ['PERSON', 'GPE', 'LOC', 'ORG', 'LANGUAGE', 'NAT_REL_POL', 'DATETIME', 'PERIOD', 'QUANTITY', 'MONEY', 'NUMERIC', 'ORDINAL', 'FACILITY', 'WORK_OF_ART', 'EVENT']

In [ ]:
def run_inference(model, dataloader, device="cuda"):
    """
    Mirrors the evaluate script's validation_step logic but
    stores per-sentence results instead of aggregating them.
    Returns a list of dicts: {tokens, gold_tags, pred_tags}
    """
    model.eval()
    records = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids, attention_mask, labels, batch_token_idx = batch
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)

            logits = model(input_ids, attention_mask)   # (B, seq_len, num_labels)
            preds  = torch.argmax(logits, dim=-1).cpu().numpy()
            labels = labels.cpu().numpy()

            for sent_preds, sent_labels, token_idx in zip(preds, labels, batch_token_idx):
                # token_idx: list of subword positions for each word
                word_preds  = [sent_preds[idx]  for idx in token_idx]
                word_labels = [sent_labels[idx]  for idx in token_idx]

                # Decode integer ids → tag strings
                pred_tags = [label_list[i] for i in word_preds]
                gold_tags = [label_list[i] for i in word_labels]

                records.append({
                    "pred_tags": pred_tags,
                    "gold_tags": gold_tags,
                })

    return records

In [ ]:
with open(datasets_path + 'diac/test.json', "r", encoding="utf8") as f:
    test_diac = json.load(f)
with open(datasets_path + 'nodiac/test.json', "r", encoding="utf8") as f:
    test_nodiac = json.load(f)

In [ ]:
def eval_model(model_name, model):
    for eval_cond, test_dataset in [("diac", test_diac), ("nodiac", test_nodiac)]:
        tokenizer = AutoTokenizer.from_pretrained(model_name, strip_accents=False)
        collator = Collator(tokenizer=tokenizer, max_seq_len=args.model_max_length)

        test_loader = DataLoader(test_dataset, batch_size=batch_size, num_workers=4, shuffle=False,
                                     collate_fn=collator, pin_memory=True)


        records = run_inference(model, test_loader)

        # ── Save raw predictions ────────────────────────────────
        pred_path = f"{PRED_DIR}/{RUN_NAME}_eval_{eval_cond}.json"
        with open(pred_path, "w", encoding="utf-8") as f:
            json.dump(records, f, ensure_ascii=False, indent=2)

        # ── Per-class seqeval report ────────────────────────────
        golds = [r["gold_tags"] for r in records]
        preds = [r["pred_tags"] for r in records]

        report = classification_report(golds, preds, output_dict=True, zero_division=0)
        import pandas as pd
        df = pd.DataFrame(report).T
        df.to_csv(f"{RES_DIR}/{RUN_NAME}_eval_{eval_cond}_per_class.csv")
        print(f"\n=== {RUN_NAME} → eval {eval_cond} ===")
        print(df.to_string())
        return df, records

In [ ]:
rows = []

df, m_diac_records = eval_model(m_model_name, m_diac_model)
row = df.loc["micro avg"] if "micro avg" in df.index else df.loc["weighted avg"]
rows.append({"run": 'm_diac', "eval": 'diac',
                 "P": round(row["precision"],4),
                 "R": round(row["recall"],4),
                 "F1": round(row["f1-score"],4)})


df, m_nodiac_records = eval_model(m_model_name, m_nodiac_model)
row = df.loc["micro avg"] if "micro avg" in df.index else df.loc["weighted avg"]
rows.append({"run": 'm_nodiac', "eval": 'nodiac',
                 "P": round(row["precision"],4),
                 "R": round(row["recall"],4),
                 "F1": round(row["f1-score"],4)})


df, ro_diac_records = eval_model(ro_model_name, ro_diac_model)
row = df.loc["micro avg"] if "micro avg" in df.index else df.loc["weighted avg"]
rows.append({"run": 'ro_diac', "eval": 'diac',
                 "P": round(row["precision"],4),
                 "R": round(row["recall"],4),
                 "F1": round(row["f1-score"],4)})

df, ro_nodiac_records = eval_model(ro_model_name, ro_nodiac_model)
row = df.loc["micro avg"] if "micro avg" in df.index else df.loc["weighted avg"]
rows.append({"run": 'ro_nodiac', "eval": 'nodiac',
                 "P": round(row["precision"],4),
                 "R": round(row["recall"],4),
                 "F1": round(row["f1-score"],4)})

summary = (pd.DataFrame(rows)
             .pivot_table(index="run", columns="eval", values=["P","R","F1"]))
print(summary.to_markdown())
summary.to_csv("4x4_matrix.csv")

# Error analysis

In [ ]:
def categorize_errors(records, sent_offset=0):
    """
    Takes the records list from run_inference (gold_tags, pred_tags per sentence).
    Returns a DataFrame with one row per span-level error.
    error_type ∈ {missed, spurious, wrong_type, wrong_boundary}
    """
    import pandas as pd

    def get_spans(tags):
        spans, i = {}, 0
        while i < len(tags):
            if tags[i].startswith("B-"):
                label = tags[i][2:]
                j = i + 1
                while j < len(tags) and tags[j] == f"I-{label}":
                    j += 1
                spans[(i, j - 1)] = label
                i = j
            else:
                i += 1
        return spans

    rows = []
    for sid, rec in enumerate(records, start=sent_offset):
        gold_spans = get_spans(rec["gold_tags"])
        pred_spans = get_spans(rec["pred_tags"])

        done_gold, done_pred = set(), set()

        # Wrong type: same (start, end), different label
        for pos in set(gold_spans) & set(pred_spans):
            gl, pl = gold_spans[pos], pred_spans[pos]
            if gl != pl:
                rows.append(dict(sent_id=sid, error_type="wrong_type",
                                 gold_class=gl, pred_class=pl,
                                 gold_span=pos, pred_span=pos))
                done_gold.add(pos); done_pred.add(pos)

        # Wrong boundary: overlapping spans with same label
        for gpos, gl in gold_spans.items():
            if gpos in done_gold: continue
            for ppos, pl in pred_spans.items():
                if ppos in done_pred: continue
                gs, ge = gpos; ps, pe = ppos
                if gl == pl and gs <= pe and ps <= ge and gpos != ppos:
                    rows.append(dict(sent_id=sid, error_type="wrong_boundary",
                                     gold_class=gl, pred_class=pl,
                                     gold_span=gpos, pred_span=ppos))
                    done_gold.add(gpos); done_pred.add(ppos); break

        # Missed (false negatives)
        for pos, label in gold_spans.items():
            if pos not in done_gold:
                rows.append(dict(sent_id=sid, error_type="missed",
                                 gold_class=label, pred_class="O",
                                 gold_span=pos, pred_span=None))

        # Spurious (false positives)
        for pos, label in pred_spans.items():
            if pos not in done_pred:
                rows.append(dict(sent_id=sid, error_type="spurious",
                                 gold_class="O", pred_class=label,
                                 gold_span=None, pred_span=pos))

    return pd.DataFrame(rows)


for records, eval_cond, run_name in zip([m_diac_records, m_nodiac_records, ro_diac_records, ro_nodiac_records], ['diac', 'nodiac', 'diac', 'nodiac'], ['m_diac', 'm_nodiac', 'ro_diac', 'ro_nodiac']):

    errors = categorize_errors(records)
    errors.to_csv(f"{RES_DIR}/{run_name}_eval_{eval_cond}_errors.csv", index=False)

    print(f"\n--- {eval_cond} error breakdown ---")
    print(errors.groupby(["gold_class", "error_type"]).size().unstack(fill_value=0))

# Attention analysis

In [ ]:
def extract_attentions(model, tokenizer, sentence, device="cuda"):
    """
    Returns (token_strings, attentions_ndarray).
    attentions shape: (num_layers, num_heads, seq_len, seq_len)
    """
    model.eval()
    enc = tokenizer(sentence, return_tensors="pt",
                    truncation=True, max_length=512).to(device)

    with torch.no_grad():
        # Pass output_attentions=True directly to the underlying BERT:
        out = model.bert(**enc, output_attentions=True)

    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    attns  = np.stack([a.squeeze(0).cpu().numpy() for a in out.attentions])
    return tokens, attns  # (num_layers, num_heads, seq, seq)


configuration = [
    {
        'records': m_diac_records,
        'eval_cond': 'diac',
        'run_name': 'm_diac',
        'model': m_diac_model,
        'model_name': m_model_name,
    },
    {
        'records': m_nodiac_records,
        'eval_cond': 'nodiac',
        'run_name': 'm_nodiac',
        'model': m_nodiac_model,
        'model_name': m_model_name,
    },
    {
        'records': ro_diac_records,
        'eval_cond': 'diac',
        'run_name': 'ro_diac',
        'model': ro_diac_model,
        'model_name': ro_model_name,
    },
    {
        'records': ro_nodiac_records,
        'eval_cond': 'nodiac',
        'run_name': 'ro_nodiac',
        'model': ro_nodiac_model,
        'model_name': ro_model_name,
    }
]

for config in configuration:
    tokenizer = AutoTokenizer.from_pretrained(config['model_name'], strip_accents=False)

    # Save attention for the first N missed entities from the diac eval:
    errors = pd.read_csv(f"{RES_DIR}/{config['run_name']}_eval_errors.csv")

    sample_ids = errors[errors["error_type"] == "missed"]["sent_id"].unique()[:30]

    for sid in sample_ids:
        rec     = config['records'][int(sid)]
        sentence = " ".join(rec["gold_tags"])  # or reconstruct from raw dataset
        tokens, attns = extract_attentions(config['model'], tokenizer, sentence)
        np.savez_compressed(
            f"{RES_DIR}/attn_sent{sid}_{config['run_name']}.npz",
            attns=attns,
            tokens=np.array(tokens, dtype=object),
        )